# Stage 03 — German Credit Data Preprocessing

This notebook creates the frozen stratified split and fits preprocessing on training predictors only. It does not train a predictive model.


## 1. Setup and frozen inputs


In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TEST_SIZE = 0.20
RAW_TARGET = "class"
TARGET = "TARGET"
EXPECTED_RAW_SHA256 = "f985b42636c9e28ce4fb3913738d1ebfd0f7947fda8d4406fe2ded5d84295acb"

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "german_credit":
    candidate = NOTEBOOK_DIR / "backend" / "ml" / "german_credit"
    if candidate.exists():
        NOTEBOOK_DIR = candidate
    else:
        raise RuntimeError("Run from backend/ml/german_credit or the project root.")
BACKEND_DIR = NOTEBOOK_DIR.parents[1]
RAW_PATH = BACKEND_DIR / "data" / "german_credit" / "raw" / "german_credit_raw.csv"
ARTIFACTS_DIR = BACKEND_DIR / "artifacts" / "german_credit"
SELECTED_PATH = ARTIFACTS_DIR / "selected_features.json"
DISPLAY_MAPPING_PATH = ARTIFACTS_DIR / "feature_display_mapping.json"
CATEGORY_MAPPING_PATH = ARTIFACTS_DIR / "category_value_mapping.json"
PROVENANCE_PATH = ARTIFACTS_DIR / "dataset_provenance.json"
RARE_CATEGORY_PATH = ARTIFACTS_DIR / "rare_category_audit.csv"


In [2]:
raw_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
raw_df = pd.read_csv(RAW_PATH)
with SELECTED_PATH.open(encoding="utf-8") as file:
    selection = json.load(file)
with DISPLAY_MAPPING_PATH.open(encoding="utf-8") as file:
    display_mapping = json.load(file)
with CATEGORY_MAPPING_PATH.open(encoding="utf-8") as file:
    category_mapping = json.load(file)
with PROVENANCE_PATH.open(encoding="utf-8") as file:
    provenance = json.load(file)

selected_features = selection["selected_features"]
numeric_features = selection["numeric_selected_features"]
categorical_features = selection["categorical_selected_features"]
excluded_features = selection["excluded_features"]
raw_features = [column for column in raw_df.columns if column != RAW_TARGET]

assert raw_sha256 == EXPECTED_RAW_SHA256
assert raw_df.shape == (1000, 21)
assert len(raw_features) == 20
assert len(selected_features) == 17
assert set(selected_features) == set(numeric_features + categorical_features)
print(f"Raw snapshot verified: {raw_sha256}")
print(f"Selected features: {len(selected_features)}")


Raw snapshot verified: f985b42636c9e28ce4fb3913738d1ebfd0f7947fda8d4406fe2ded5d84295acb
Selected features: 17


## 2. Modelling target

The original target remains unchanged. The modelling target maps good credit to 0 and bad/higher-risk credit to 1.


In [3]:
X = raw_df[selected_features].copy()
y = raw_df[RAW_TARGET].map({1: 0, 2: 1}).rename(TARGET)

target_summary = pd.DataFrame({
    "TARGET": [0, 1],
    "meaning": ["Good credit risk", "Bad / higher-risk credit"],
    "count": [int(y.eq(0).sum()), int(y.eq(1).sum())],
    "percentage": [float(y.eq(0).mean() * 100), float(y.eq(1).mean() * 100)],
})
display(target_summary)
assert set(y.unique()) == {0, 1}
assert y.value_counts().to_dict() == {0: 700, 1: 300}
assert raw_df[RAW_TARGET].value_counts().to_dict() == {1: 700, 2: 300}


,TARGET,meaning,count,percentage
0,0,Good credit risk,700,70.0
1,1,Bad / higher-risk credit,300,30.0


## 3. Frozen stratified holdout

One 80/20 split is created with `random_state=42` and target stratification. Original row indexes are retained as experimental identifiers.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_indices = pd.DataFrame({"row_index": X_train.index.astype(int)})
test_indices = pd.DataFrame({"row_index": X_test.index.astype(int)})
TRAIN_INDICES_PATH = ARTIFACTS_DIR / "train_indices.csv"
TEST_INDICES_PATH = ARTIFACTS_DIR / "test_indices.csv"

if TRAIN_INDICES_PATH.exists() or TEST_INDICES_PATH.exists():
    assert TRAIN_INDICES_PATH.exists() and TEST_INDICES_PATH.exists()
    saved_train = pd.read_csv(TRAIN_INDICES_PATH)
    saved_test = pd.read_csv(TEST_INDICES_PATH)
    assert saved_train.equals(train_indices), "Existing training indexes do not match the frozen split."
    assert saved_test.equals(test_indices), "Existing test indexes do not match the frozen split."
else:
    train_indices.to_csv(TRAIN_INDICES_PATH, index=False)
    test_indices.to_csv(TEST_INDICES_PATH, index=False)

train_index_set = set(train_indices["row_index"])
test_index_set = set(test_indices["row_index"])
assert train_index_set.isdisjoint(test_index_set)
assert train_index_set | test_index_set == set(raw_df.index)
assert len(train_index_set) + len(test_index_set) == len(raw_df)

split_summary = pd.DataFrame({
    "partition": ["training", "test"],
    "rows": [len(X_train), len(X_test)],
    "TARGET_0": [int(y_train.eq(0).sum()), int(y_test.eq(0).sum())],
    "TARGET_1": [int(y_train.eq(1).sum()), int(y_test.eq(1).sum())],
    "TARGET_1_percentage": [y_train.mean() * 100, y_test.mean() * 100],
})
display(split_summary)
assert abs(y_train.mean() - y.mean()) < 0.01
assert abs(y_test.mean() - y.mean()) < 0.01


,partition,rows,TARGET_0,TARGET_1,TARGET_1_percentage
0,training,800,560,240,30.0
1,test,200,140,60,30.0


## 4. Training-only preprocessing

Numeric values use median imputation and are not scaled. Categorical values use most-frequent imputation followed by full one-hot encoding with unknown-category handling.


In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop=None, sparse_output=False)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Training processed shape: {X_train_processed.shape}")
print(f"Test processed shape: {X_test_processed.shape}")
print(f"Output type: {type(X_train_processed).__name__}")


Training processed shape: (800, 54)
Test processed shape: (200, 54)
Output type: ndarray


## 5. Leakage checks

Learned imputation statistics and categorical levels are checked against the training partition. Test rows are transformed only.


In [6]:
fitted_numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
fitted_categorical_imputer = preprocessor.named_transformers_["categorical"].named_steps["imputer"]
fitted_encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]

training_medians = X_train[numeric_features].median().to_numpy(dtype=float)
assert np.allclose(fitted_numeric_imputer.statistics_.astype(float), training_medians)

training_modes = np.array([X_train[feature].mode(dropna=True).iloc[0] for feature in categorical_features], dtype=object)
assert np.array_equal(fitted_categorical_imputer.statistics_, training_modes)

train_imputed_categories = fitted_categorical_imputer.transform(X_train[categorical_features])
test_imputed_categories = fitted_categorical_imputer.transform(X_test[categorical_features])
test_only_rows = []
for position, feature in enumerate(categorical_features):
    learned = set(fitted_encoder.categories_[position])
    training_observed = set(pd.unique(train_imputed_categories[:, position]))
    test_observed = set(pd.unique(test_imputed_categories[:, position]))
    assert learned == training_observed
    for value in sorted(test_observed - learned):
        test_only_rows.append({
            "feature": feature,
            "display_feature": display_mapping[feature],
            "category_value": value,
            "category_label": category_mapping[feature].get(str(value), str(value)),
            "test_count": int((X_test[feature] == value).sum()),
        })
test_only_categories = pd.DataFrame(
    test_only_rows,
    columns=["feature", "display_feature", "category_value", "category_label", "test_count"],
)

print(f"Numeric medians verified from training: {len(training_medians)}")
print(f"Categorical modes verified from training: {len(training_modes)}")
print(f"Categories present only in test: {len(test_only_categories)}")
display(test_only_categories)
assert TARGET not in X_train.columns and RAW_TARGET not in X_train.columns


Numeric medians verified from training: 6
Categorical modes verified from training: 11
Categories present only in test: 0


,feature,display_feature,category_value,category_label,test_count


## 6. Transformed feature mapping

The mapping is built from the fitted encoder categories, preserving the link from every encoded column to its raw feature and UCI category label.


In [7]:
transformed_names = preprocessor.get_feature_names_out().tolist()
mapping_rows = []
for feature in numeric_features:
    mapping_rows.append({
        "transformed_feature": f"numeric__{feature}",
        "original_feature": feature,
        "display_feature": display_mapping[feature],
        "feature_type": "numeric",
        "category_value": "",
        "category_label": "",
    })
for feature, categories in zip(categorical_features, fitted_encoder.categories_):
    for category in categories:
        mapping_rows.append({
            "transformed_feature": f"categorical__{feature}_{category}",
            "original_feature": feature,
            "display_feature": display_mapping[feature],
            "feature_type": "categorical_one_hot",
            "category_value": category,
            "category_label": category_mapping[feature][str(category)],
        })
transformed_mapping = pd.DataFrame(mapping_rows)
assert transformed_mapping["transformed_feature"].tolist() == transformed_names
assert len(transformed_mapping) == X_train_processed.shape[1]
TRANSFORMED_MAPPING_PATH = ARTIFACTS_DIR / "transformed_feature_mapping.csv"
transformed_mapping.to_csv(TRANSFORMED_MAPPING_PATH, index=False)
display(transformed_mapping)


,transformed_feature,original_feature,display_feature,feature_type,category_value,category_label
0,numeric__Attribute2,Attribute2,Credit duration (months),numeric,,
1,numeric__Attribute5,Attribute5,Credit amount (DM),numeric,,
2,numeric__Attribute8,Attribute8,Installment rate (% disposable income),numeric,,
3,numeric__Attribute11,Attribute11,Residence duration,numeric,,
4,numeric__Attribute16,Attribute16,Existing credits at bank,numeric,,
5,numeric__Attribute18,Attribute18,Dependants,numeric,,
6,categorical__Attribute1_A11,Attribute1,Checking account status,categorical_one_hot,A11,< 0 DM
7,categorical__Attribute1_A12,Attribute1,Checking account status,categorical_one_hot,A12,0 to < 200 DM
8,categorical__Attribute1_A13,Attribute1,Checking account status,categorical_one_hot,A13,>= 200 DM or salary assignment >= 1 year
9,categorical__Attribute1_A14,Attribute1,Checking account status,categorical_one_hot,A14,No checking account


## 7. Dimensionality and processed-data checks


In [8]:
actual_transformed_count = X_train_processed.shape[1]
dimensionality = pd.DataFrame({
    "measure": [
        "Raw selected features", "Numeric features", "Categorical features",
        "Transformed features", "Training rows", "Test rows",
    ],
    "value": [
        len(selected_features), len(numeric_features), len(categorical_features),
        actual_transformed_count, len(X_train), len(X_test),
    ],
})
display(dimensionality)
print(f"Stage 02 estimate: approximately 54 transformed features")
print(f"Actual training-derived count: {actual_transformed_count}")

def matrix_values(matrix):
    return matrix.data if sparse.issparse(matrix) else np.asarray(matrix)

train_values = matrix_values(X_train_processed)
test_values = matrix_values(X_test_processed)
train_nan_count = int(np.isnan(train_values).sum())
test_nan_count = int(np.isnan(test_values).sum())
train_nonfinite_count = int((~np.isfinite(train_values)).sum())
test_nonfinite_count = int((~np.isfinite(test_values)).sum())
assert train_nan_count == 0 and test_nan_count == 0
assert train_nonfinite_count == 0 and test_nonfinite_count == 0
print(f"Remaining train NaNs: {train_nan_count}")
print(f"Remaining test NaNs: {test_nan_count}")


,measure,value
0,Raw selected features,17
1,Numeric features,6
2,Categorical features,11
3,Transformed features,54
4,Training rows,800
5,Test rows,200


Stage 02 estimate: approximately 54 transformed features
Actual training-derived count: 54
Remaining train NaNs: 0
Remaining test NaNs: 0


## 8. Rare-category split audit

Stage 02 rare levels are counted separately in the frozen training and test partitions. No categories are merged.


In [9]:
stage02_rare = pd.read_csv(RARE_CATEGORY_PATH)
rare_split_rows = []
for row in stage02_rare.itertuples(index=False):
    training_count = int(X_train[row.raw_feature].eq(row.raw_value).sum())
    test_count = int(X_test[row.raw_feature].eq(row.raw_value).sum())
    rare_split_rows.append({
        "feature": row.raw_feature,
        "category": row.raw_value,
        "category_label": row.category_label,
        "full_data_count": int(row.count),
        "training_count": training_count,
        "test_count": test_count,
        "present_in_training": training_count > 0,
        "present_in_test": test_count > 0,
    })
rare_category_split_audit = pd.DataFrame(rare_split_rows)
assert (rare_category_split_audit["training_count"] + rare_category_split_audit["test_count"] == rare_category_split_audit["full_data_count"]).all()
RARE_SPLIT_PATH = ARTIFACTS_DIR / "rare_category_split_audit.csv"
rare_category_split_audit.to_csv(RARE_SPLIT_PATH, index=False)
display(rare_category_split_audit)


,feature,category,category_label,full_data_count,training_count,test_count,present_in_training,present_in_test
0,Attribute4,A48,Retraining,9,5,4,True,True
1,Attribute4,A44,Domestic appliances,12,10,2,True,True
2,Attribute4,A410,Other,12,6,6,True,True


## 9. Conservative DiCE actionability metadata

Only credit duration, amount and installment rate are permitted candidates for future variation. DiCE is not configured or executed here.


In [10]:
DIRECTLY_ACTIONABLE = {
    "Attribute2": "Requested credit duration can be adjusted in an application scenario.",
    "Attribute5": "Requested credit amount can be adjusted.",
    "Attribute8": "Installment rate is linked to loan design and affordability.",
}
actionability_rows = []
for feature in selected_features:
    if feature in DIRECTLY_ACTIONABLE:
        status = "directly_actionable_candidate"
        reason = DIRECTLY_ACTIONABLE[feature]
    else:
        status = "not_permitted_to_vary"
        reason = "Conservative default; not approved as a direct counterfactual intervention."
    actionability_rows.append({
        "feature": feature,
        "display_feature": display_mapping[feature],
        "actionability_status": status,
        "reason": reason,
    })
dice_actionability = pd.DataFrame(actionability_rows)
DICE_ACTIONABILITY_PATH = ARTIFACTS_DIR / "dice_actionability_candidates.csv"
dice_actionability.to_csv(DICE_ACTIONABILITY_PATH, index=False)
display(dice_actionability)


,feature,display_feature,actionability_status,reason
0,Attribute1,Checking account status,not_permitted_to_vary,Conservative default; not approved as a direct...
1,Attribute2,Credit duration (months),directly_actionable_candidate,Requested credit duration can be adjusted in a...
2,Attribute3,Credit history,not_permitted_to_vary,Conservative default; not approved as a direct...
3,Attribute4,Credit purpose,not_permitted_to_vary,Conservative default; not approved as a direct...
4,Attribute5,Credit amount (DM),directly_actionable_candidate,Requested credit amount can be adjusted.
5,Attribute6,Savings account status,not_permitted_to_vary,Conservative default; not approved as a direct...
6,Attribute7,Employment duration,not_permitted_to_vary,Conservative default; not approved as a direct...
7,Attribute8,Installment rate (% disposable income),directly_actionable_candidate,Installment rate is linked to loan design and ...
8,Attribute10,Other debtors or guarantors,not_permitted_to_vary,Conservative default; not approved as a direct...
9,Attribute11,Residence duration,not_permitted_to_vary,Conservative default; not approved as a direct...


## 10. Save the frozen preprocessor and metadata

Processed matrices are not saved because they can be reproduced from the raw snapshot, frozen indexes and preprocessor.


In [11]:
PREPROCESSOR_PATH = ARTIFACTS_DIR / "preprocessor.joblib"
METADATA_PATH = ARTIFACTS_DIR / "preprocessing_metadata.json"
joblib.dump(preprocessor, PREPROCESSOR_PATH)

def distribution(series):
    counts = series.value_counts().sort_index()
    percentages = series.value_counts(normalize=True).sort_index().mul(100)
    return {
        str(int(value)): {"count": int(counts[value]), "percentage": float(percentages[value])}
        for value in counts.index
    }

preprocessing_metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_mapping": {"1": 0, "2": 1},
    "selected_features": selected_features,
    "excluded_features": excluded_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "training_target_distribution": distribution(y_train),
    "test_target_distribution": distribution(y_test),
    "transformed_feature_count": actual_transformed_count,
    "output_representation": "dense numpy array",
    "imputation_strategy": {"numeric": "median", "categorical": "most_frequent"},
    "one_hot_configuration": {"handle_unknown": "ignore", "drop": None, "min_frequency": None, "sparse_output": False},
    "raw_snapshot": str(RAW_PATH.relative_to(BACKEND_DIR)),
    "raw_snapshot_sha256": raw_sha256,
    "provenance_reference": str(PROVENANCE_PATH.relative_to(BACKEND_DIR)),
    "train_indices": str(TRAIN_INDICES_PATH.relative_to(BACKEND_DIR)),
    "test_indices": str(TEST_INDICES_PATH.relative_to(BACKEND_DIR)),
    "test_only_categories": test_only_categories.to_dict(orient="records"),
    "rare_categories_merged": False,
    "numeric_scaling": False,
    "validation_design": {
        "frozen_holdout": {"test_size": TEST_SIZE, "stratified": True},
        "training_only_model_development": {"method": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": RANDOM_STATE},
        "uci_cost_matrix_use": "additional Stage 04 evaluation measure",
    },
    "predictive_model_trained": False,
}
with METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(preprocessing_metadata, file, indent=2)

print(f"Preprocessor saved: {PREPROCESSOR_PATH}")
print(f"Metadata saved: {METADATA_PATH}")


Preprocessor saved: C:\Users\H P E L I T E\Desktop\XAI_CreditStudy\backend\artifacts\german_credit\preprocessor.joblib
Metadata saved: C:\Users\H P E L I T E\Desktop\XAI_CreditStudy\backend\artifacts\german_credit\preprocessing_metadata.json


## 11. Integrity checks and summary


In [12]:
assert len(raw_df) == 1000
assert len(selected_features) == 17
assert len(X_train) == 800 and len(X_test) == 200
assert int(y.sum()) == 300 and int(y.eq(0).sum()) == 700
assert not train_index_set.intersection(test_index_set)
assert all(feature not in selected_features for feature in ["Attribute9", "Attribute13", "Attribute20"])
assert np.allclose(fitted_numeric_imputer.statistics_.astype(float), X_train[numeric_features].median().to_numpy(dtype=float))
assert train_nan_count == 0 and test_nan_count == 0
assert PREPROCESSOR_PATH.exists()
assert TRAIN_INDICES_PATH.exists() and TEST_INDICES_PATH.exists()
assert TRANSFORMED_MAPPING_PATH.exists()

print(f"Raw rows: {len(raw_df)}")
print(f"Selected features: {len(selected_features)}")
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Training TARGET 1 percentage: {y_train.mean() * 100:.1f}%")
print(f"Test TARGET 1 percentage: {y_test.mean() * 100:.1f}%")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Transformed features: {actual_transformed_count}")
print(f"Training processed shape: {X_train_processed.shape}")
print(f"Test processed shape: {X_test_processed.shape}")
print(f"Remaining train NaNs: {train_nan_count}")
print(f"Remaining test NaNs: {test_nan_count}")
print(f"Train/test overlap: {len(train_index_set.intersection(test_index_set))}")
print(f"Preprocessor saved: {PREPROCESSOR_PATH.exists()}")
print("Predictive model trained: False")
print("SHAP executed: False")
print("LIME executed: False")
print("DiCE executed: False")


Raw rows: 1000
Selected features: 17
Training rows: 800
Test rows: 200
Training TARGET 1 percentage: 30.0%
Test TARGET 1 percentage: 30.0%
Numeric features: 6
Categorical features: 11
Transformed features: 54
Training processed shape: (800, 54)
Test processed shape: (200, 54)
Remaining train NaNs: 0
Remaining test NaNs: 0
Train/test overlap: 0


Preprocessor saved: True
Predictive model trained: False
SHAP executed: False
LIME executed: False
DiCE executed: False


## 12. Stage 04 review points

Review the frozen indexes, the training-derived category set, the rare-level audit and the 54-column representation. The planned Stage 04 design is five-fold stratified validation within training data, followed by one frozen-holdout evaluation and an additional UCI cost measure.

Stage 04 has not been created.
